# Data Investigation — image / mask / IP overlays by cohort

Browse an nnUNet dataset's `imagesTr` + `labelsTr` to check that masks (and insertion points)
sit correctly on the image. Splits into **SmartHealth** vs **DirVsAverages** cohorts so you can
compare (used to debug why Dataset102 = SmartHealth+DirVsAvg deteriorates in training).

Set `DATASET_ID` and run. Colours match `eval_util.plot_original_with_masks`:
LV = yellow, IP1 = red, IP2 = blue.

In [ ]:
import os, glob, re
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

RAW = '/home/sastocke/nnUNet/nnUNet_raw'
DATASET_ID = 102          # <-- change to investigate any dataset
CHANNEL    = 0            # which contrast to show underneath (0=avg, 1=MD, 2=E1, 3=FA)

D = glob.glob(f'{RAW}/Dataset{DATASET_ID:03d}_*')[0]
IMG, LAB = f'{D}/imagesTr', f'{D}/labelsTr'
print('dataset:', os.path.basename(D))

def cohort(name):
    n = name.lower()
    return 'DirVsAvg' if ('dirvsavg' in n or 'dirvsaverages' in n) else 'SmartHealth'

cases = sorted({re.sub(r'_\d{4}\.nii\.gz$', '', os.path.basename(f)) for f in glob.glob(f'{IMG}/*.nii.gz')})
groups = {'SmartHealth': [], 'DirVsAvg': []}
for c in cases: groups[cohort(c)].append(c)
print('cases per cohort:', {k: len(v) for k, v in groups.items()})

# label colours: 0 transparent, 1 yellow (LV), 2 red (IP1), 3 blue (IP2)
MASK_CMAP = ListedColormap([(0,0,0,0), (1,1,0,0.5), (1,0,0,0.6), (0,0,1,0.6)])

def load(c, ch=CHANNEL):
    img = np.asanyarray(nib.load(f'{IMG}/{c}_{ch:04d}.nii.gz').dataobj).squeeze().astype(float)
    m   = np.asanyarray(nib.load(f'{LAB}/{c}.nii.gz').dataobj).squeeze()
    return img, m

## 1. Overlay montage — image | mask | overlay (per cohort)
The red/yellow ring must sit **on the myocardium** in the overlay column. Compare DirVsAvg
against the known-good SmartHealth rows.

In [ ]:
def show_cohort(coh, n=6, ch=CHANNEL):
    cs = groups[coh]
    if not cs:
        print(f'(no {coh} cases)'); return
    picks = cs[::max(1, len(cs)//n)][:n]
    fig, ax = plt.subplots(len(picks), 3, figsize=(9, 3*len(picks)))
    if len(picks) == 1: ax = ax[None, :]
    for r, c in enumerate(picks):
        img, m = load(c, ch)
        ax[r,0].imshow(img, cmap='gray')
        ax[r,1].imshow(m, cmap='nipy_spectral')
        ax[r,2].imshow(img, cmap='gray')
        ax[r,2].imshow(np.ma.masked_where(m==0, m), cmap=MASK_CMAP, vmin=0, vmax=3, interpolation='none')
        ax[r,0].set_ylabel(c.replace('DirVsAvgHannum_','DVA_').replace('Hannum_Volunteer_','V')[:30], fontsize=7)
        for k in range(3): ax[r,k].set_xticks([]); ax[r,k].set_yticks([])
    ax[0,0].set_title(f'{coh}: ch{ch} image'); ax[0,1].set_title('mask'); ax[0,2].set_title('overlay')
    plt.tight_layout(); plt.show()

show_cohort('SmartHealth', n=6)   # known-good reference
show_cohort('DirVsAvg',    n=8)   # suspects

## 2. Quantitative alignment proxy (flip / transpose check)
Mean image intensity inside the mask vs the mask flipped up-down / left-right / transposed.
If a flipped mask matches the image **better** than `as-is`, the masks are misaligned.

In [ ]:
def align_scores(coh, n=12, ch=CHANNEL):
    cs = groups[coh][::max(1, len(groups[coh])//n)][:n]
    print(f'== {coh} == (ch{ch} mean inside mask; as-is should be the intended alignment)')
    for c in cs:
        img, m = load(c, ch); fg = m > 0
        if fg.sum() == 0:
            print(f'  {c[:44]:44s} EMPTY MASK'); continue
        s = dict(asis=img[fg].mean(), ud=img[np.flipud(fg)].mean(),
                 lr=img[np.fliplr(fg)].mean(),
                 T=(img[fg.T].mean() if fg.shape[0]==fg.shape[1] else float('nan')))
        print(f'  {c[:44]:44s} ' + '  '.join(f'{k}={v:.3f}' for k,v in s.items()))

align_scores('SmartHealth')
print()
align_scores('DirVsAvg')

## 3. Insertion-point check (only if this is an IP dataset, labels 2/3 present)
Marks GT insertion points (label 2 = red +, label 3 = blue x) on the image.

In [ ]:
from scipy.ndimage import center_of_mass
def show_ips(coh, n=6, ch=CHANNEL):
    cs = [c for c in groups[coh] if set(np.unique(load(c)[1])) & {2,3}]
    if not cs:
        print(f'({coh}: no IP labels 2/3 in this dataset)'); return
    picks = cs[::max(1, len(cs)//n)][:n]
    fig, ax = plt.subplots(1, len(picks), figsize=(3*len(picks), 3))
    if len(picks)==1: ax=[ax]
    for a, c in zip(ax, picks):
        img, m = load(c, ch); a.imshow(img, cmap='gray')
        for lbl, mk, col in ((2,'+','red'), (3,'x','blue')):
            if np.any(m==lbl):
                y,x = center_of_mass(m==lbl); a.scatter(x,y,marker=mk,c=col,s=90,linewidths=2)
        a.set_title(c.replace('DirVsAvgHannum_','DVA_').replace('Hannum_Volunteer_','V')[:22], fontsize=7)
        a.axis('off')
    plt.tight_layout(); plt.show()

show_ips('SmartHealth'); show_ips('DirVsAvg')